## 1. Project NER schema

Our eventual structured incident pipeline needs information that can support humanitarian response and later GIS/intelligence layers.

| Entity | Purpose |
|---|---|
| `LOCATION` | place names and geographical references |
| `CASUALTY` | deaths, injuries, missing/trapped casualty expressions |
| `DISPLACED` | evacuation/displacement references |
| `REQUEST` | expressed requests for assistance |
| `RESOURCE` | supplies or aid resources |
| `RESCUE` | rescue/emergency-operation references |
| `DISASTER_TYPE` | named disaster/hazard |
| `ORGANIZATION` | responding agencies/organizations |
| `PERSON` | named people |
| `NUMBER` | standalone quantities/numbers |

### Project extension

`INFRASTRUCTURE` is intentionally included in our **target project schema**, because later we need to represent damage such as bridges, roads, hospitals, houses/buildings and power lines.

If the verified dataset does not contain this entity type, we will **not invent labels**. We will either map a suitable existing resource or perform a controlled custom annotation stage.


In [1]:
# ============================================================
# 2. Environment setup
# ============================================================

!pip -q install -U \
    "transformers==4.44.2" \
    "tokenizers==0.19.1" \
    "datasets>=2.20,<3.0" \
    "seqeval>=1.2.2" \
    "huggingface_hub>=0.24,<1.0" \
    "pandas>=2.0" \
    "numpy<2.1"

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from huggingface_hub import list_datasets

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Environment ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Environment ready.
PyTorch: 2.10.0+cu128
CUDA available: True


In [2]:
# ============================================================
# 3. Project taxonomy
# ============================================================

BASE_ENTITY_TYPES = [
    "LOCATION",
    "CASUALTY",
    "DISPLACED",
    "REQUEST",
    "RESOURCE",
    "RESCUE",
    "DISASTER_TYPE",
    "ORGANIZATION",
    "PERSON",
    "NUMBER",
]

PROJECT_EXTENSION_TYPES = [
    "INFRASTRUCTURE",
]

PROJECT_ENTITY_TYPES = BASE_ENTITY_TYPES + PROJECT_EXTENSION_TYPES

print("Base entity types:")
for x in BASE_ENTITY_TYPES:
    print(" -", x)

print("\nProject extension types:")
for x in PROJECT_EXTENSION_TYPES:
    print(" -", x)

print("\nTotal project entity types:", len(PROJECT_ENTITY_TYPES))


Base entity types:
 - LOCATION
 - CASUALTY
 - DISPLACED
 - REQUEST
 - RESOURCE
 - RESCUE
 - DISASTER_TYPE
 - ORGANIZATION
 - PERSON
 - NUMBER

Project extension types:
 - INFRASTRUCTURE

Total project entity types: 11


## 4. Discover disaster-specific NER datasets

We do not hard-code an unverified Hugging Face dataset ID.

The Hugging Face Hub is queried for datasets matching terms such as `HUMAID-NER`, `disaster NER`, and `crisis NER`.

The notebook will show the candidates that actually exist at runtime.


In [3]:
# ============================================================
# 4. Search Hugging Face Hub
# ============================================================

SEARCH_TERMS = [
    "HUMAID-NER",
    "disaster NER",
    "crisis NER",
]

candidates = {}

for term in SEARCH_TERMS:
    print(f"\nSearching: {term}")
    try:
        results = list(list_datasets(search=term, limit=20))
        for item in results:
            dataset_id = getattr(item, "id", None)
            if dataset_id:
                candidates[dataset_id] = item
                print("  ", dataset_id)
    except Exception as e:
        print("  Search failed:", repr(e))

print("\nUnique candidates found:", len(candidates))

candidate_ids = sorted(candidates.keys())
candidate_ids[:50]



Searching: HUMAID-NER

Searching: disaster NER
   NerdyPie/Disaster_Management
   Ahzy181/vietnamese-NER-disaster

Searching: crisis NER

Unique candidates found: 2


['Ahzy181/vietnamese-NER-disaster', 'NerdyPie/Disaster_Management']

## 5. Select the dataset

### Preferred resource

**HUMAID-NER** is especially relevant because it was released as an NER extension of HumAID and uses disaster-specific entities such as `LOCATION`, `CASUALTY`, `DISPLACED`, `REQUEST`, `RESOURCE`, and `RESCUE`.

However, its published annotations were produced with a hybrid automatic labelling pipeline rather than 60,000 tweets being manually annotated. Therefore we will treat it as a **candidate training resource**, not automatically as perfect ground truth.

The notebook searches the discovered candidates for an exact/near-exact HUMAID-NER match.

If no HUMAID-NER dataset is available on the Hub at runtime, the notebook stops cleanly and tells us to use the published dataset/code or another verified source rather than silently switching datasets.


In [4]:
# ============================================================
# 5. Resolve HUMAID-NER
# ============================================================

preferred = [
    dataset_id for dataset_id in candidate_ids
    if "humaid" in dataset_id.lower() and "ner" in dataset_id.lower()
]

print("HUMAID-NER candidates:")
for dataset_id in preferred:
    print(" -", dataset_id)

if len(preferred) == 0:
    print(
        "\nNo HUMAID-NER dataset was discoverable through the Hugging Face Hub "
        "at this moment. DO NOT train yet."
    )
    print(
        "We will use the verified public release/manual dataset source after "
        "confirming its exact repository."
    )
    SELECTED_DATASET_ID = None
else:
    SELECTED_DATASET_ID = preferred[0]
    print("\nSelected candidate:", SELECTED_DATASET_ID)


HUMAID-NER candidates:

No HUMAID-NER dataset was discoverable through the Hugging Face Hub at this moment. DO NOT train yet.
We will use the verified public release/manual dataset source after confirming its exact repository.


In [5]:
# ============================================================
# 6. Load selected dataset safely
# ============================================================

dataset = None

if SELECTED_DATASET_ID is not None:
    try:
        dataset = load_dataset(
            SELECTED_DATASET_ID,
            verification_mode="no_checks",
        )
        print(dataset)
        print("\nAvailable splits:", list(dataset.keys()))
    except Exception as e:
        print("Dataset loading failed:")
        print(repr(e))
        dataset = None


## 7. Inspect dataset structure

Different NER datasets use different field names.

Common possibilities include:

- `tokens`
- `ner_tags`
- `text`
- `labels`
- `tags`

We therefore inspect the actual schema instead of assuming a `train/dev/test` layout.


In [6]:
# ============================================================
# 7. Inspect splits and columns
# ============================================================

if dataset is not None:

    for split_name, split_data in dataset.items():
        print("=" * 70)
        print("SPLIT:", split_name)
        print("ROWS:", len(split_data))
        print("COLUMNS:", split_data.column_names)
        print("FEATURES:")
        print(split_data.features)
        print()

else:
    print("Dataset is not loaded; no schema inspection performed.")


Dataset is not loaded; no schema inspection performed.


In [7]:
# ============================================================
# 8. Detect NER columns
# ============================================================

def detect_column(columns, preferred_names):
    lower_map = {c.lower(): c for c in columns}

    for name in preferred_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    for c in columns:
        cl = c.lower()
        for name in preferred_names:
            if name.lower() in cl:
                return c

    return None


if dataset is not None:

    first_split = next(iter(dataset.keys()))
    columns = dataset[first_split].column_names

    TOKEN_COLUMN = detect_column(
        columns,
        ["tokens", "token", "words", "word"]
    )

    LABEL_COLUMN = detect_column(
        columns,
        ["ner_tags", "ner_tag", "labels", "tags", "tag"]
    )

    TEXT_COLUMN = detect_column(
        columns,
        ["tweet_text", "text", "sentence"]
    )

    print("First split:", first_split)
    print("Token column:", TOKEN_COLUMN)
    print("Label column:", LABEL_COLUMN)
    print("Text column:", TEXT_COLUMN)

else:
    first_split = None
    TOKEN_COLUMN = None
    LABEL_COLUMN = None
    TEXT_COLUMN = None


## 9. Inspect label names

For a BIO NER dataset, we expect labels resembling:

```text
O
B-LOCATION
I-LOCATION
B-CASUALTY
I-CASUALTY
...
```

The actual labels are taken from the dataset rather than manually assumed.


In [8]:
# ============================================================
# 9. Inspect label names
# ============================================================

label_names = None

if dataset is not None and LABEL_COLUMN is not None:

    feature = dataset[first_split].features[LABEL_COLUMN]

    print("Label feature:", feature)

    if hasattr(feature, "names"):
        label_names = list(feature.names)

    elif hasattr(feature, "feature") and hasattr(feature.feature, "names"):
        label_names = list(feature.feature.names)

    print("\nDetected label names:")
    if label_names:
        for i, label in enumerate(label_names):
            print(f"{i:>3}: {label}")
    else:
        print("Could not automatically recover label names from this feature.")

else:
    print("Label metadata cannot be inspected because the dataset is not ready.")


Label metadata cannot be inspected because the dataset is not ready.


In [9]:
# ============================================================
# 10. Extract entity types from BIO labels
# ============================================================

def entity_types_from_bio(label_names):
    entity_types = set()

    for label in label_names or []:
        if label == "O":
            continue

        if "-" in label:
            prefix, entity = label.split("-", 1)
            if prefix in {"B", "I"}:
                entity_types.add(entity)

    return sorted(entity_types)


dataset_entity_types = entity_types_from_bio(label_names)

print("Dataset entity types:")
for x in dataset_entity_types:
    print(" -", x)

print("\nProject entity types:")
for x in PROJECT_ENTITY_TYPES:
    print(" -", x)

missing_from_dataset = [
    x for x in PROJECT_ENTITY_TYPES
    if x not in dataset_entity_types
]

print("\nProject entities NOT covered by candidate dataset:")
for x in missing_from_dataset:
    print(" -", x)


Dataset entity types:

Project entity types:
 - LOCATION
 - CASUALTY
 - DISPLACED
 - REQUEST
 - RESOURCE
 - RESCUE
 - DISASTER_TYPE
 - ORGANIZATION
 - PERSON
 - NUMBER
 - INFRASTRUCTURE

Project entities NOT covered by candidate dataset:
 - LOCATION
 - CASUALTY
 - DISPLACED
 - REQUEST
 - RESOURCE
 - RESCUE
 - DISASTER_TYPE
 - ORGANIZATION
 - PERSON
 - NUMBER
 - INFRASTRUCTURE


## 11. Label distribution

NER labels are usually highly imbalanced because `O` appears much more often than entity tokens.

We inspect both:

1. BIO-label frequency
2. entity-type frequency

This helps identify rare entities before training.


In [10]:
# ============================================================
# 11. Label distribution
# ============================================================

if dataset is not None and LABEL_COLUMN is not None:

    for split_name, split_data in dataset.items():

        counter = Counter()

        for row in split_data:
            tags = row[LABEL_COLUMN]

            if isinstance(tags, list):
                if label_names and tags and isinstance(tags[0], int):
                    tags = [
                        label_names[i] if 0 <= i < len(label_names) else str(i)
                        for i in tags
                    ]

                counter.update(tags)

        print("=" * 70)
        print("SPLIT:", split_name)

        for label, count in counter.most_common():
            print(f"{label:25s} {count:,}")

else:
    print("Dataset not ready for distribution analysis.")


Dataset not ready for distribution analysis.


## 12. Inspect real annotated examples

This is a critical quality-control step.

We need to visually inspect examples before accepting the dataset.

If annotations look wrong, we do **not** proceed directly to model training.


In [11]:
# ============================================================
# 12. Display annotated examples
# ============================================================

def convert_tags(tags):
    if label_names and tags and isinstance(tags[0], int):
        return [
            label_names[i] if 0 <= i < len(label_names) else str(i)
            for i in tags
        ]
    return tags


def display_bio_example(tokens, tags):
    tags = convert_tags(tags)

    print("-" * 90)

    for token, tag in zip(tokens, tags):
        print(f"{token:25s} {tag}")

    print("-" * 90)


if dataset is not None and TOKEN_COLUMN and LABEL_COLUMN:

    sample_split = next(iter(dataset.keys()))
    sample_data = dataset[sample_split]

    for i in range(min(10, len(sample_data))):
        row = sample_data[i]

        print(f"\nEXAMPLE {i}")
        display_bio_example(
            row[TOKEN_COLUMN],
            row[LABEL_COLUMN]
        )

else:
    print("No token-level dataset available for example inspection.")


No token-level dataset available for example inspection.


## 13. Split verification

We do not assume the split names.

Acceptable patterns include:

```text
train / validation / test
train / dev / test
train / val / test
```

The notebook reports what actually exists.


In [12]:
# ============================================================
# 13. Split verification
# ============================================================

if dataset is not None:

    split_names = list(dataset.keys())

    print("Actual splits:")
    for name in split_names:
        print(f" - {name}: {len(dataset[name]):,} rows")

    normalized = {name.lower(): name for name in split_names}

    TRAIN_SPLIT = normalized.get("train")
    TEST_SPLIT = normalized.get("test")

    VALIDATION_SPLIT = (
        normalized.get("validation")
        or normalized.get("dev")
        or normalized.get("val")
    )

    print("\nResolved:")
    print("TRAIN:", TRAIN_SPLIT)
    print("VALIDATION:", VALIDATION_SPLIT)
    print("TEST:", TEST_SPLIT)

else:
    TRAIN_SPLIT = VALIDATION_SPLIT = TEST_SPLIT = None


## 14. Leakage / duplicate check

Because our classification work already uses HumAID-derived disaster tweets, we need to be careful about evaluation leakage.

We compare text where possible.

This is a **reporting check**, not a blind zero-duplicate assertion. Social-media datasets can contain retweets and repeated content.


In [13]:
# ============================================================
# 14. Cross-split duplicate check
# ============================================================

def get_text_set(split_data, text_column, token_column):
    values = set()

    if split_data is None:
        return values

    for row in split_data:
        if text_column and text_column in row and row[text_column] is not None:
            values.add(str(row[text_column]).strip())

        elif token_column and token_column in row and row[token_column] is not None:
            values.add(" ".join(map(str, row[token_column])).strip())

    return values


if dataset is not None:

    split_text_sets = {}

    for split_name, split_data in dataset.items():
        split_text_sets[split_name] = get_text_set(
            split_data,
            TEXT_COLUMN,
            TOKEN_COLUMN
        )

    split_list = list(split_text_sets.keys())

    for i in range(len(split_list)):
        for j in range(i + 1, len(split_list)):

            a = split_list[i]
            b = split_list[j]

            overlap = split_text_sets[a].intersection(
                split_text_sets[b]
            )

            print(
                f"{a} ↔ {b}: "
                f"{len(overlap):,} exact text overlaps"
            )

else:
    print("Dataset not available for leakage check.")


Dataset not available for leakage check.


## 15. Dataset suitability decision

We now score the candidate against the requirements.

### Required

- disaster-domain English text
- token-level/BIO NER annotations
- train/validation/test or equivalent splits
- useful disaster/humanitarian entities
- inspectable labels
- manageable for Kaggle training

### Important methodological warning

HUMAID-NER's published methodology describes a **hybrid automatic annotation process**. That makes it useful, but we should report it accurately:

> **training data uses automatically generated / weakly supervised disaster-domain NER annotations unless we independently validate or manually correct a subset.**

We should not call those annotations "fully human-annotated" unless we verify that independently.


In [14]:
# ============================================================
# 15. Automatic suitability report
# ============================================================

report = {
    "dataset_loaded": dataset is not None,
    "dataset_id": SELECTED_DATASET_ID,
    "has_train": TRAIN_SPLIT is not None,
    "has_validation": VALIDATION_SPLIT is not None,
    "has_test": TEST_SPLIT is not None,
    "has_token_column": TOKEN_COLUMN is not None,
    "has_label_column": LABEL_COLUMN is not None,
    "dataset_entity_types": dataset_entity_types,
    "missing_project_entities": missing_from_dataset,
}

print(json.dumps(report, indent=2))

print("\n" + "=" * 70)
print("PRELIMINARY DECISION")
print("=" * 70)

if not report["dataset_loaded"]:
    print("STOP: dataset was not verified. Do not train yet.")

elif not report["has_train"] or not report["has_test"]:
    print("STOP: required train/test splits are missing.")

elif not report["has_token_column"] or not report["has_label_column"]:
    print("STOP: token-level NER fields were not verified.")

else:
    print("Candidate is structurally suitable for further inspection.")
    if missing_from_dataset:
        print(
            "\nIMPORTANT: the candidate does not cover every project entity."
        )
        print(
            "We will handle missing entities through controlled annotation/"
            "supplementary data rather than inventing labels."
        )
    else:
        print("Candidate covers all current project entities.")


{
  "dataset_loaded": false,
  "dataset_id": null,
  "has_train": false,
  "has_validation": false,
  "has_test": false,
  "has_token_column": false,
  "has_label_column": false,
  "dataset_entity_types": [],
  "missing_project_entities": [
    "LOCATION",
    "CASUALTY",
    "DISPLACED",
    "REQUEST",
    "RESOURCE",
    "RESCUE",
    "DISASTER_TYPE",
    "ORGANIZATION",
    "PERSON",
    "NUMBER",
    "INFRASTRUCTURE"
  ]
}

PRELIMINARY DECISION
STOP: dataset was not verified. Do not train yet.


In [15]:
# ============================================================
# 16. Persist verification report to disk
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase4_ner_verification")
RESULTS_DIR = OUTPUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

full_report = dict(report)  # base report from the suitability-decision step

full_report["search_terms"] = SEARCH_TERMS
full_report["candidates_found"] = candidate_ids
full_report["humaid_ner_candidates"] = preferred
full_report["resolved_splits"] = {
    "train": TRAIN_SPLIT,
    "validation": VALIDATION_SPLIT,
    "test": TEST_SPLIT,
}
full_report["token_column"] = TOKEN_COLUMN
full_report["label_column"] = LABEL_COLUMN
full_report["text_column"] = TEXT_COLUMN
full_report["label_names"] = label_names
full_report["project_entity_types"] = PROJECT_ENTITY_TYPES

report_path = RESULTS_DIR / "phase4_dataset_verification_report.json"
report_path.write_text(json.dumps(full_report, indent=2), encoding="utf-8")

print(f"Saved verification report: {report_path}")
print("\nUse this file as the evidence record for the Phase 4 dataset decision")
print("(Path A / B / C) before starting the NER training notebook.")


Saved verification report: /kaggle/working/phase4_ner_verification/results/phase4_dataset_verification_report.json

Use this file as the evidence record for the Phase 4 dataset decision
(Path A / B / C) before starting the NER training notebook.


## 17. Final output

After running this notebook, we should have enough evidence to make the next decision.

### Path A — verified dataset is usable

```text
Dataset verification
        ↓
annotation quality audit
        ↓
BIO preprocessing
        ↓
NER model
```

### Path B — dataset is useful but incomplete

```text
HUMAID-NER
    +
controlled supplementary annotation
    ↓
unified project NER dataset
    ↓
NER model
```

### Path C — dataset is not accessible/suitable

```text
Do not force it
    ↓
verify another disaster NER resource
    ↓
or create controlled custom annotations
```

**Next notebook after this one:** NER training and evaluation, only after the dataset decision is confirmed.
